# LangChain Tools

- Tools are a way to add more functionality to your LLM

## Simple LLM create

In [25]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=2,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

responce = model.stream("Hello, how are you ?")

for i in responce:
    print(i.text, end="", flush=True)

Hello! I'm doing great, thanks for asking. How can I help you today?

### Tools Create

In [26]:
# LangChain Tools

from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """
    Get the weather for a location.
    """
    return f"The weather in {location} is sunny." 

### Tool Binding with `model`

In [27]:
model_with_tools = model.bind_tools([get_weather])

In [28]:
responce = model_with_tools.invoke("What is the weather in Serampore?")
responce

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in Serampore?" We need to get weather, using provided function get_weather. We\'ll call function.', 'tool_calls': [{'id': 'fc_890f3068-63b1-45d6-9e55-1d1b711170c1', 'function': {'arguments': '{"location":"Serampore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 129, 'total_tokens': 189, 'completion_time': 0.140986747, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.004877726, 'prompt_tokens_details': None, 'queue_time': 0.053410254, 'total_time': 0.145864473}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bf073-3fdb-7132-bec8-5993cffaf9dd-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Serampore'}, 'id': 'fc_890f3068-63b1-45

In [29]:
responce.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Serampore'},
  'id': 'fc_890f3068-63b1-45d6-9e55-1d1b711170c1',
  'type': 'tool_call'}]

## Tool Execution Loop

### Step 1: Model Genates Tool Call

In [30]:
massage = [{"role": "user", "content": "What is the weather in Serampore?"}]

ai_message = model_with_tools.invoke(massage)
massage.append(ai_message)
massage

[{'role': 'user', 'content': 'What is the weather in Serampore?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in Serampore?" We need to get weather for that location. There\'s a function get_weather. We should call it with location "Serampore".', 'tool_calls': [{'id': 'fc_2f563536-3762-4611-b8c2-444bbc8ce249', 'function': {'arguments': '{"location":"Serampore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 129, 'total_tokens': 199, 'completion_time': 0.149097743, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.005141565, 'prompt_tokens_details': None, 'queue_time': 0.156546105, 'total_time': 0.154239308}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bf073-411b-7391-9292-531b957db

In [31]:
ai_message.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Serampore'},
  'id': 'fc_2f563536-3762-4611-b8c2-444bbc8ce249',
  'type': 'tool_call'}]

### Step 2: Execute tools and collect results

In [46]:
for tool_call in ai_message.tool_calls:
    # Execute the tool with the genarated arguments
    tool_result = get_weather.invoke(tool_call)
    massage.append(tool_result)
massage

[{'role': 'user', 'content': 'What is the weather in Serampore?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in Serampore?" We need to get weather for that location. There\'s a function get_weather. We should call it with location "Serampore".', 'tool_calls': [{'id': 'fc_2f563536-3762-4611-b8c2-444bbc8ce249', 'function': {'arguments': '{"location":"Serampore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 129, 'total_tokens': 199, 'completion_time': 0.149097743, 'completion_tokens_details': {'reasoning_tokens': 41}, 'prompt_time': 0.005141565, 'prompt_tokens_details': None, 'queue_time': 0.156546105, 'total_time': 0.154239308}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bf073-411b-7391-9292-531b957db

### Step 3: Pass the result back to the model for the final result

In [39]:
final_result = model_with_tools.invoke(massage)
final_result.content

'The current weather in Serampore is **sunny**. Enjoy the bright, clear skies!'